# HERMES on Google Colab

This notebook runs `train_hermes.py` on a Colab GPU and stores `data/`, `checkpoints/`, and `runs/` on Google Drive so runs can resume across runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/golf-colab'
REPO_URL = 'https://github.com/ahmedshakill/golf.git'
GITHUB_USER = 'ahmedshakill'
GITHUB_EMAIL = 'shakil.000024@gmail.com'
GITHUB_TOKEN = ''  # Optional: set a fine-scoped token if you want notebook-driven push.
AUTO_PUSH_AFTER_TRAIN = False
COMMIT_MESSAGE = 'Update from Colab'
REPO_DIR = '/content/golf'
RUN_NAME = 'hermes_v1_fineweb_ode8'
TRAIN_PATH = 'data/fineweb_train.bin'
VAL_PATH = 'data/fineweb_val.bin'
BATCH_SIZE = 192
TRAIN_STEPS = 10000
ODE_STEPS = 8
NUM_WORKERS = 2
USE_ACTIVATION_CHECKPOINTING = False
USE_COMPILE = False

print('DRIVE_ROOT =', DRIVE_ROOT)
print('REPO_URL   =', REPO_URL)
print('REPO_DIR   =', REPO_DIR)
print('RUN_NAME   =', RUN_NAME)

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q datasets huggingface_hub pyarrow

In [ ]:
import os
import pathlib
import shutil
import subprocess
from urllib.parse import urlsplit

for path in [DRIVE_ROOT, f'{DRIVE_ROOT}/data', f'{DRIVE_ROOT}/checkpoints', f'{DRIVE_ROOT}/runs']:
    pathlib.Path(path).mkdir(parents=True, exist_ok=True)

def authed_repo_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    parsed = urlsplit(repo_url)
    return f'{parsed.scheme}://{GITHUB_USER}:{token}@{parsed.netloc}{parsed.path}'

clone_url = authed_repo_url(REPO_URL, GITHUB_TOKEN)

if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(['git', 'clone', clone_url, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'config', 'user.name', GITHUB_USER], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'config', 'user.email', GITHUB_EMAIL], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', REPO_URL], check=True)

for name in ['data', 'checkpoints', 'runs']:
    repo_path = os.path.join(REPO_DIR, name)
    drive_path = os.path.join(DRIVE_ROOT, name)
    if os.path.islink(repo_path):
        continue
    if os.path.isdir(repo_path):
        shutil.rmtree(repo_path)
    elif os.path.exists(repo_path):
        os.remove(repo_path)
    os.symlink(drive_path, repo_path)

print('Repo ready at', REPO_DIR)
print('Drive-backed data at', DRIVE_ROOT)

In [ ]:
import subprocess
from urllib.parse import urlsplit

def git_commit_and_push_if_needed(commit_message=COMMIT_MESSAGE):
    if not GITHUB_TOKEN:
        print('Skipping push: GITHUB_TOKEN is empty.')
        return

    status = subprocess.run(
        ['git', '-C', REPO_DIR, 'status', '--porcelain'],
        check=True,
        capture_output=True,
        text=True,
    )
    if not status.stdout.strip():
        print('No local changes to commit.')
        return

    parsed = urlsplit(REPO_URL)
    push_url = f'{parsed.scheme}://{GITHUB_USER}:{GITHUB_TOKEN}@{parsed.netloc}{parsed.path}'
    subprocess.run(['git', '-C', REPO_DIR, 'add', '-A'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'commit', '-m', commit_message], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'push', push_url, 'main'], check=True)
    print('Pushed to GitHub.')

In [ ]:
%cd /content/golf
!git status --short
!git pull --ff-only origin main

In [ ]:
%cd /content/golf
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    if hasattr(torch.cuda, 'is_bf16_supported'):
        print('bf16 supported:', torch.cuda.is_bf16_supported())

## Optional: prepare FineWeb slices in Colab

Skip this cell if the dataset already exists on Drive. For a T4, prefer `.bin` files over plain text.

In [ ]:
%cd /content/golf
!python3 prepare_fineweb.py --subset CC-MAIN-2024-10 --format bin --train-mb 64 --val-mb 8

## Train or resume

This Colab profile is tuned for a 16 GB T4: `float16`, bin datasets, a larger batch, more loader workers, and no activation checkpointing by default.

In [ ]:
%cd /content/golf
import subprocess

train_cmd = [
    'python3', 'train_hermes.py',
    '--run_name', RUN_NAME,
    '--train_path', TRAIN_PATH,
    '--val_path', VAL_PATH,
    '--batch', str(BATCH_SIZE),
    '--steps', str(TRAIN_STEPS),
    '--ode_steps', str(ODE_STEPS),
    '--dtype', 'float16',
    '--num_workers', str(NUM_WORKERS),
    '--checkpoint_dir', 'checkpoints',
    '--results_file', 'runs/hermes_runs.jsonl',
    '--resume', 'latest',
]

if not USE_ACTIVATION_CHECKPOINTING:
    train_cmd.append('--no_activation_checkpointing')
if USE_COMPILE:
    train_cmd.append('--compile')

print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, check=True)

In [ ]:
%cd /content/golf
!find checkpoints -maxdepth 2 -type f | sort | tail -n 10
!tail -n 5 runs/hermes_runs.jsonl

if AUTO_PUSH_AFTER_TRAIN:
    git_commit_and_push_if_needed()

## Optional: commit and push notebook-side changes

This only works if `GITHUB_TOKEN` is set at the top of the notebook.

In [ ]:
git_commit_and_push_if_needed()